# HEART — Duplicate Impact Evidence (v2, read-only)

**Research question:** Does removing repeated raw-SMILES records materially change the source chemistry distribution, scaffold exposure, or molecular 3D conformer evidence?

This extends the earlier quick test, reusing the **completed full raw SMILES audit** (18,837,320 training records, 13,270,476 unique raw identities), and builds **slide-ready quantitative evidence**. It does **not** train HEART, modify LMDBs, submit jobs, or claim that duplicate removal will improve downstream accuracy.

**New in v2:** independent actual uniform-unique sampling (not just inverse-frequency reweighting), paired bootstrap differences/95% intervals, descriptor CDF comparison, Murcko-scaffold diversity, direct sample of duplicate records' 3D coordinates using LMDB numeric keys, final progress-slide summary. All work is read-only and runs in your current Jupyter kernel; no GPU required.

**Definition:** "Deduplicated" means one molecular identity per *raw SMILES*; no canonical SMILES or stereochemistry collapse is applied. Duplicates may have distinct stored conformers. Results are descriptive, not a proof of equal performance or a replacement training set.


## 0 — Paths and safety checks

In [ ]:
from pathlib import Path
import json, hashlib, pickle, time
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 17
N_RAW = 10000
N_UNIQUE = 5000

HOME = Path('/home/ypan/heart_liver/model_train')
PROJECT = Path('/project/yli4/ypan')
SCRATCH = Path('/scratch/ypan/heart_liver/model_train')

# Override if your paths differ; notebook does not search the entire filesystem.
DATA_OVERRIDE = None
AUDIT_OVERRIDE = None
RESULTS_OVERRIDE = None

DATA_CANDIDATES = [
    PROJECT/'UniMolReference/ligand_data/ligands',
    HOME/'UniMolReference/ligand_data/ligands',
    SCRATCH/'UniMolReference/ligand_data/ligands',
]
AUDIT_CANDIDATES = [
    HOME/'heart_evaluation/general_audit_10k_v2/full_ligand_audit/raw',
    PROJECT/'heart_evaluation/general_audit_10k_v2/full_ligand_audit/raw',
    SCRATCH/'heart_evaluation/general_audit_10k_v2/full_ligand_audit/raw',
]

DATA = Path(DATA_OVERRIDE) if DATA_OVERRIDE else next(
    (p for p in DATA_CANDIDATES if (p/'train.lmdb').is_file()), DATA_CANDIDATES[0])
AUDIT = Path(AUDIT_OVERRIDE) if AUDIT_OVERRIDE else next(
    (p for p in AUDIT_CANDIDATES if (p/'train_raw_hashes.txt').is_file()), AUDIT_CANDIDATES[0])
RESULTS = (Path(RESULTS_OVERRIDE) if RESULTS_OVERRIDE else
           HOME/'heart_evaluation/duplicate_impact_expanded_evidence')

TRAIN_LMDB = DATA/'train.lmdb'
RAW_HASHES = AUDIT/'train_raw_hashes.txt'
UNIQUE_HASHES = AUDIT/'train_raw_unique_sorted.txt'
SUMMARY = AUDIT/'audit_summary_raw.json'
EXPORT = AUDIT/'train_raw_export.json'

for name,p in [('train LMDB',TRAIN_LMDB),('raw hash export',RAW_HASHES),
               ('unique sorted hashes',UNIQUE_HASHES),('audit summary',SUMMARY),('audit provenance',EXPORT)]:
    print(f'{name:22} exists={p.is_file()}  {p}')
missing=[p for p in (TRAIN_LMDB,RAW_HASHES,UNIQUE_HASHES,SUMMARY,EXPORT) if not p.is_file()]
if missing:
    raise FileNotFoundError('Missing existing audit assets: '+', '.join(map(str,missing))+
                            '. Do not rerun 18.8M audit; locate its existing output and set AUDIT_OVERRIDE above.')

full=json.loads(SUMMARY.read_text())
ex=json.loads(EXPORT.read_text())
assert full['method']=='raw' and not full['train']['limited'], 'Requires COMPLETED full raw audit'
assert ex['identity']=='raw' and not ex['limited'], 'This is a partial or non-raw hash export'
assert full['train']['total_records_scanned']==18_837_320, 'Unexpected training dataset size'
assert int(ex['usable_records'])==full['train']['usable_records']
expected_path=Path(ex['path']).resolve()
if expected_path!=TRAIN_LMDB.resolve():
    raise RuntimeError(f'Audit hashes refer to {expected_path}, but current DATA is {TRAIN_LMDB.resolve()}. '
                       'Do not mix different LMDB versions. Restore the matching source or prove file equivalence first.')
assert RAW_HASHES.stat().st_size==33*ex['usable_records'], 'Raw-hash file is incomplete'
assert int(ex['usable_records']) == int(ex['total_records_scanned']) if 'total_records_scanned' in ex else int(ex['usable_records']) == int(full['train']['total_records_scanned']), 'Cannot map hash-export positions to lexicographic LMDB numeric keys if export omitted records'
assert UNIQUE_HASHES.stat().st_size==33*full['train']['unique_identifiers'], 'Unique-hash file is incomplete'
print('\nFull audit verified: total',f"{full['train']['total_records_scanned']:,}",
      'raw-unique',f"{full['train']['unique_identifiers']:,}",
      'duplicate excess',f"{full['train']['duplicate_record_excess']:,}")
RESULTS.mkdir(parents=True,exist_ok=True)
print('NEW outputs only:',RESULTS)

## 1 — Random raw records + random **unique molecule identities**

The independent unique-ID sample comes from `train_raw_unique_sorted.txt`, not from deduplicating a 1.5K mini-sample (which would be misleading when there are 13M identities).

In [ ]:
import lmdb
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
RDLogger.DisableLog('rdApp.error')

def hash_smi(s):
    return hashlib.blake2b(s.strip().encode('utf8'),digest_size=16).hexdigest()

# The hashed file is a fixed-width ASCII table: 32-hex-digit digest plus newline.
# Seek by exact line offset, so sampling unique molecules does not load 13M hashes into RAM.
rng=np.random.default_rng(SEED)
unique_n=int(full['train']['unique_identifiers'])
unique_line_idx=np.sort(rng.choice(unique_n,size=min(N_UNIQUE,unique_n),replace=False))
unique_hashes=[]
with UNIQUE_HASHES.open('rb') as f:
    for i in unique_line_idx:
        f.seek(33*int(i))
        entry=f.read(33)
        if len(entry)!=33 or entry[32:33]!=b'\n':
            raise ValueError('Unexpected hash index format')
        unique_hashes.append(entry[:32].decode('ascii'))
assert len(set(unique_hashes))==len(unique_hashes)

# Actual raw records: sample numeric LMDB keys 0..N-1 (NOT cursor ordinal positions).
env=lmdb.open(str(TRAIN_LMDB),subdir=False,readonly=True,lock=False,
              readahead=False,meminit=False,max_readers=128)
raw_rows=[]
try:
    with env.begin(write=False) as txn:
        n_lmdb=txn.stat()['entries']
        assert n_lmdb==full['train']['total_records_scanned'], (n_lmdb,full['train']['total_records_scanned'])
        raw_keys=rng.choice(n_lmdb,size=min(N_RAW,n_lmdb),replace=False)
        for j,k in enumerate(raw_keys,1):
            v=txn.get(str(int(k)).encode('ascii'))
            if v is None:raise KeyError(f'LMDB numeric key missing: {k}')
            d=pickle.loads(v)  # Trusted local Uni-Mol LMDB; read-only.
            smi=d.get('smi',d.get(b'smi'))
            if isinstance(smi,bytes):smi=smi.decode('utf-8')
            if not isinstance(smi,str) or not smi.strip():
                continue
            smi=smi.strip()
            mol=Chem.MolFromSmiles(smi)
            raw_rows.append(dict(
                lmdb_key=int(k),smiles=smi,digest=hash_smi(smi),
                conformer_slots=len(d.get('coordinates',[])),
                heavy_atoms=mol.GetNumHeavyAtoms() if mol else np.nan,
                molecular_weight=Descriptors.MolWt(mol) if mol else np.nan,
                logP=Descriptors.MolLogP(mol) if mol else np.nan,
                TPSA=rdMolDescriptors.CalcTPSA(mol) if mol else np.nan,
                HBD=rdMolDescriptors.CalcNumHBD(mol) if mol else np.nan,
                HBA=rdMolDescriptors.CalcNumHBA(mol) if mol else np.nan,
                rotatable_bonds=rdMolDescriptors.CalcNumRotatableBonds(mol) if mol else np.nan,
                ring_count=rdMolDescriptors.CalcNumRings(mol) if mol else np.nan,
                murcko_scaffold=(MurckoScaffold.MurckoScaffoldSmiles(mol=mol) or '[acyclic]') if mol else None,
            ))
            if j%500==0:print('Random numeric LMDB records sampled:',j,'/',len(raw_keys),flush=True)
finally:
    env.close()
raw=pd.DataFrame(raw_rows)
if raw.empty:raise RuntimeError('No valid SMILES in sampled records')
print('Raw records sampled:',len(raw),'; separately sampled distinct hashed molecules:',len(unique_hashes))
print('Conformer slots per record (raw sample):',raw.conformer_slots.value_counts().head().to_dict())

## 2 — Count global occurrence for the sampled molecules

One **sequential pass over the existing 18.84M-line hash text export**, not over the full LMDB. Reports exact full-data copy count for each randomly chosen molecule.

In [ ]:
targets=set(raw['digest'])|set(unique_hashes)
# Needed for independent unique sample and a small record-pair conformer check.
# Hash file was exported by an LMDB cursor (lexicographic ASCII numeric key order).
unique_targets=set(unique_hashes)
first_unique_export_rank={}
PAIR_CANDIDATES=set(raw.drop_duplicates('digest').sample(
    n=min(700,len(raw.drop_duplicates('digest'))),random_state=SEED)['digest'])
max_pair_positions=2
pair_export_ranks={}
freq={key:0 for key in targets}
t0=time.monotonic()
scanned=0
with RAW_HASHES.open('rb',buffering=8*1024*1024) as f:
    for line in f:
        scanned+=1
        h=line[:32].decode('ascii')
        if h in freq:freq[h]+=1
        if h in unique_targets and h not in first_unique_export_rank:
            first_unique_export_rank[h]=scanned-1
        if h in PAIR_CANDIDATES:
            slots=pair_export_ranks.setdefault(h,[])
            if len(slots)<max_pair_positions: slots.append(scanned-1)
        if scanned%5_000_000==0:
            print(f'Existing hashes scanned {scanned:,} / {ex["usable_records"]:,}',flush=True)
assert scanned==int(ex['usable_records']), 'Raw audit export truncated'
assert min(freq.values())>0, 'A selected digest was not found. Check path & audit provenance.'
raw['copies_in_full_train']=raw['digest'].map(freq).astype('int64')
uniq=pd.DataFrame({'digest':unique_hashes})
uniq['copies_in_full_train']=uniq['digest'].map(freq).astype('int64')
print('Sampled frequencies ready; elapsed',round(time.monotonic()-t0,1),'s')
print('Raw-record sample: median copies',raw.copies_in_full_train.median(),
      '; fraction belonging to repeated identities',round((raw.copies_in_full_train>1).mean()*100,2),'%')
print('Deduplicated-identity sample: median copies in original file',uniq.copies_in_full_train.median(),
      '; fraction of unique identities repeated in original',round((uniq.copies_in_full_train>1).mean()*100,2),'%')
assert len(first_unique_export_rank)==len(unique_hashes), 'Missing independent-unique export positions'


## 3 — Raw vs deduplicated: summary and figures

**Interpretation:** raw sampling is proportional to the number of rows per SMILES; deduplicated sampling gives each distinct raw SMILES equal probability. The same molecules are not being re-inferred by HEART here.

In [ ]:
def binned(v):
    b=[0,1,2,3,5,10,20,np.inf]
    labels=['1','2','3','4–5','6–10','11–20','>20']
    return pd.cut(v,bins=b,labels=labels,include_lowest=True)
raw_hist=binned(raw.copies_in_full_train).value_counts(normalize=True).sort_index()
uniq_hist=binned(uniq.copies_in_full_train).value_counts(normalize=True).sort_index()
comparison=pd.DataFrame({'Raw record sampling (%)':100*raw_hist,
                         'Dedup unique molecule sampling (%)':100*uniq_hist}).fillna(0)
print('Copy-count distribution — full dataset frequency for sampled molecules:')
display(comparison.round(2))
summary=pd.DataFrame([
    dict(sampling='Raw random records',sample_n=len(raw),
         pct_repeat_identity=100*(raw.copies_in_full_train>1).mean(),
         median_full_copy_count=raw.copies_in_full_train.median(),
         p95_full_copy_count=raw.copies_in_full_train.quantile(.95),
         max_full_copy_count_in_sample=raw.copies_in_full_train.max()),
    dict(sampling='Uniform dedup unique molecules',sample_n=len(uniq),
         pct_repeat_identity=100*(uniq.copies_in_full_train>1).mean(),
         median_full_copy_count=uniq.copies_in_full_train.median(),
         p95_full_copy_count=uniq.copies_in_full_train.quantile(.95),
         max_full_copy_count_in_sample=uniq.copies_in_full_train.max()),
])
display(summary.round(2))

fig,ax=plt.subplots(figsize=(9,4.5))
pos=np.arange(len(comparison))
ax.bar(pos-.19,comparison['Raw record sampling (%)'].to_numpy(),.38,label='Raw record sampling')
ax.bar(pos+.19,comparison['Dedup unique molecule sampling (%)'].to_numpy(),.38,label='Uniform unique molecules')
ax.set_xticks(pos,comparison.index.astype(str));ax.set_xlabel('Number of records per raw SMILES in FULL training LMDB')
ax.set_ylabel('Share of sampled observations (%)')
ax.set_title('How duplication changes molecular exposure')
ax.legend();ax.grid(axis='y',alpha=.2)
plt.tight_layout()
fig.savefig(RESULTS/'01_raw_vs_dedup_copy_frequency.png',dpi=190)
fig.savefig(RESULTS/'01_raw_vs_dedup_copy_frequency.pdf')
plt.show()

summary.to_csv(RESULTS/'frequency_sample_summary.csv',index=False)
comparison.to_csv(RESULTS/'frequency_histogram.csv')
raw.to_csv(RESULTS/'random_raw_records_with_global_frequency.csv',index=False)
uniq.to_csv(RESULTS/'random_unique_hashes_with_global_frequency.csv',index=False)
print('Output:',RESULTS)

## 4 — Chemistry descriptor reweighting (original quick test)

First run the original inverse-copy-frequency analysis, now using up to 10K raw records. Later we independently fetch actual random unique molecules to confirm its estimates. Similar histograms alone cannot establish equivalence or rule out effects on rare scaffolds.


In [ ]:
properties=['heavy_atoms','molecular_weight','logP','TPSA','HBD','HBA','rotatable_bonds','ring_count']
rows=[]
w=1.0/raw.copies_in_full_train.to_numpy(dtype=float)
for name in properties:
    x=raw[name].to_numpy(dtype=float)
    ok=np.isfinite(x)
    if not ok.any():continue
    raw_mean=float(np.mean(x[ok]))
    dedup_mean_est=float(np.average(x[ok],weights=w[ok]))
    rows.append(dict(property=name,raw_record_mean=raw_mean,
                     dedup_unique_mean_est=dedup_mean_est,
                     difference=dedup_mean_est-raw_mean,n_usable=int(ok.sum())))
chemical=pd.DataFrame(rows)
display(chemical.round(3))
chemical.to_csv(RESULTS/'chemical_descriptor_reweight_preview.csv',index=False)

fig,axes=plt.subplots(1,2,figsize=(11,4))
for ax,feature in zip(axes,['heavy_atoms','logP']):
    x=raw[feature].to_numpy(dtype=float)
    ok=np.isfinite(x)
    if not ok.any():continue
    lo,hi=np.quantile(x[ok],[.01,.99]);bins=np.linspace(lo,hi,21)
    ax.hist(x[ok],bins=bins,weights=np.ones(ok.sum())/ok.sum(),
            alpha=.5,label='Raw records')
    weights=w[ok]/w[ok].sum()
    ax.hist(x[ok],bins=bins,weights=weights,alpha=.5,
            label='Estimated uniform unique')
    ax.set(title=f'Sample-weighted {feature} distribution',xlabel=feature,ylabel='Proportion')
    ax.legend()
plt.tight_layout()
fig.savefig(RESULTS/'02_chemistry_raw_vs_dedup_preview.png',dpi=190)
fig.savefig(RESULTS/'02_chemistry_raw_vs_dedup_preview.pdf')
plt.show()

report={
    'purpose':'Random duplicate-impact comparison; NO model training or model accuracy evaluation',
    'identity':'raw SMILES (BLAKE2b-128)',
    'full_train_records':int(full['train']['total_records_scanned']),
    'full_train_unique_raw_smiles':int(full['train']['unique_identifiers']),
    'full_duplicate_record_excess':int(full['train']['duplicate_record_excess']),
    'full_duplicate_record_excess_fraction':float(full['train']['duplicate_record_excess_fraction']),
    'n_raw_random_records':len(raw),'n_independent_unique_hashes':len(uniq),'random_seed':SEED,
    'raw_sample_fraction_of_records_with_repeated_identity':float((raw.copies_in_full_train>1).mean()),
    'unique_sample_fraction_of_identifiers_repeated_in_original':float((uniq.copies_in_full_train>1).mean()),
    'caveat':'Same-SMILES records can contain different conformers; this test does not measure their independent conformer diversity, nor the impact of retraining on deduplicated data.'
}
(RESULTS/'report.json').write_text(json.dumps(report,indent=2)+'\n')
print('SAVED: plots, CSV, and report.json to',RESULTS)
print('IMPORTANT: If these distributions differ strongly, it supports a balanced-sampling ablation,')
print('but it does NOT prove that a retrained deduplicated HEART will outperform the current checkpoint.')

## 5 — True independent uniform-unique molecule sample (no second LMDB scan)

Instead of treating inverse-frequency weighted raw records as the only "dedup" estimate, this retrieves the **actual randomly selected distinct raw-SMILES identities** sampled from the 13.27M-line unique-hash file.

The audit exported `train_raw_hashes.txt` by ascending LMDB cursor key (**lexicographic order**, not numeric); the code below converts those *cursor ordinals* back to ASCII integer LMDB keys without walking the LMDB. Every retrieved SMILES is hash-verified. **This mapping requires 100% usable raw export**, asserted earlier.


In [ ]:
# Exact lexicographic unranking of ASCII integer LMDB keys b'0'..b'(N-1)'.
# No leading-zero keys, and b'0' has no children.
def count_prefix(prefix, n):
    if prefix == '0': return 1 if n > 0 else 0
    if prefix.startswith('0'): return 0
    p = int(prefix); lo = p; hi = p + 1; total = 0
    while lo < n:
        total += max(0, min(n, hi) - lo)
        lo *= 10; hi *= 10
    return total

def lex_rank_to_integer(rank, n):
    rank = int(rank)
    if not 0 <= rank < n: raise IndexError(rank)
    prefix = ''
    while True:
        if prefix:
            if rank == 0: return int(prefix)
            rank -= 1  # the prefix itself precedes its descendants
            if prefix == '0': raise ValueError('Unexpected 0 prefix child')
        for digit in '0123456789':
            p = prefix + digit
            c = count_prefix(p, n)
            if rank < c:
                prefix = p
                break
            rank -= c
        else: raise AssertionError('Lexicographic ordinal resolution failed')

# Exhaustive small-fixture test: never blindly conflate cursor ordinal with numeric key.
for n in (1,2,9,10,11,19,100,101,1234):
    expected=sorted(map(str,range(n)))
    got=[str(lex_rank_to_integer(r,n)) for r in range(n)]
    assert expected==got,(n,expected[:10],got[:10])
print('Lexicographic rank mapping: synthetic validation PASS')

# A real independent uniform-unique sample: 5K random distinct hashes by default.
real_unique_rows=[]
env=lmdb.open(str(TRAIN_LMDB),subdir=False,readonly=True,lock=False,
              readahead=False,meminit=False,max_readers=128)
try:
    with env.begin(write=False) as txn:
        n_lmdb=txn.stat()['entries']
        for j,h in enumerate(unique_hashes,1):
            numeric_key=lex_rank_to_integer(first_unique_export_rank[h],n_lmdb)
            v=txn.get(str(numeric_key).encode('ascii'))
            assert v is not None,('missing numeric key',numeric_key)
            d=pickle.loads(v)
            smi=d.get('smi',d.get(b'smi'))
            if isinstance(smi,bytes): smi=smi.decode('utf8')
            assert isinstance(smi,str) and hash_smi(smi)==h, (
                'hash-export and LMDB order mismatch',numeric_key,h)
            mol=Chem.MolFromSmiles(smi)
            if mol is None: continue
            real_unique_rows.append(dict(
                lmdb_key=numeric_key,digest=h,smiles=smi,
                copies_in_full_train=int(freq[h]),
                heavy_atoms=mol.GetNumHeavyAtoms(),
                molecular_weight=Descriptors.MolWt(mol),
                logP=Descriptors.MolLogP(mol),
                TPSA=rdMolDescriptors.CalcTPSA(mol),
                HBD=rdMolDescriptors.CalcNumHBD(mol),
                HBA=rdMolDescriptors.CalcNumHBA(mol),
                rotatable_bonds=rdMolDescriptors.CalcNumRotatableBonds(mol),
                ring_count=rdMolDescriptors.CalcNumRings(mol),
                murcko_scaffold=MurckoScaffold.MurckoScaffoldSmiles(mol=mol) or '[acyclic]'))
            if j%1000==0:print('Independent unique identities fetched',j,'/',len(unique_hashes),flush=True)
finally:
    env.close()
unique_real=pd.DataFrame(real_unique_rows)
assert not unique_real.empty
assert unique_real['digest'].nunique()==len(unique_real)
unique_real.to_csv(RESULTS/'independent_true_unique_sample.csv',index=False)
print('Actual independent unique molecules with valid RDKit structures:',len(unique_real), '/',len(unique_hashes))

## 6 — Direct raw vs deduplicated chemistry: uncertainty, effect size, CDF shape

For each descriptor, compare **two independently sampled groups**: randomly sampled raw LMDB records vs randomly sampled unique identities. Bootstrap intervals describe sampling uncertainty for the *distribution differences*, not proof that models trained on the two sources would be equivalent. Also show an inverse-frequency estimate from the raw group as a consistency check.

In [ ]:
from scipy.stats import ks_2samp
FEATURES=['heavy_atoms','molecular_weight','logP','TPSA','HBD','HBA','rotatable_bonds','ring_count']
N_BOOT=400
rng_ci=np.random.default_rng(SEED+102)
stats_rows=[]
for feat in FEATURES:
    a=raw[feat].to_numpy(float);a=a[np.isfinite(a)]
    b=unique_real[feat].to_numpy(float);b=b[np.isfinite(b)]
    if min(len(a),len(b))<30: continue
    boot_a=np.mean(a[rng_ci.integers(0,len(a),(N_BOOT,len(a)))],axis=1)
    boot_b=np.mean(b[rng_ci.integers(0,len(b),(N_BOOT,len(b)))],axis=1)
    boot_diff=boot_b-boot_a
    delta=float(b.mean()-a.mean())
    sd=float(a.std(ddof=1))
    ks=float(ks_2samp(a,b).statistic)
    stats_rows.append(dict(
        descriptor=feat,n_raw=len(a),n_dedup=len(b),raw_mean=a.mean(),
        unique_mean=b.mean(),difference_unique_minus_raw=delta,
        difference_ci_low=np.quantile(boot_diff,.025),
        difference_ci_high=np.quantile(boot_diff,.975),
        standardised_difference=delta/sd if sd>0 else np.nan,
        raw_sd=sd,KS_statistic=ks))
chemistry_direct=pd.DataFrame(stats_rows)
display(chemistry_direct.round(4))
chemistry_direct.to_csv(RESULTS/'descriptor_direct_comparison_bootstrap.csv',index=False)

fig,axes=plt.subplots(2,2,figsize=(12,8))
for ax,feat in zip(axes.ravel(),['heavy_atoms','logP','TPSA','rotatable_bonds']):
    a=raw[feat].dropna().to_numpy(float)
    b=unique_real[feat].dropna().to_numpy(float)
    lo=np.quantile(np.r_[a,b],.005); hi=np.quantile(np.r_[a,b],.995)
    bins=np.linspace(lo,hi,28) if hi>lo else np.linspace(lo-.5,hi+.5,10)
    ax.hist(a,bins=bins,density=True,alpha=.48,label='Raw records')
    ax.hist(b,bins=bins,density=True,alpha=.48,label='Actual dedup identities')
    metric=chemistry_direct.set_index('descriptor').loc[feat]
    ax.set(title=f'{feat}: Δ={metric.difference_unique_minus_raw:+.3g}, KS={metric.KS_statistic:.3f}',
           xlabel=feat,ylabel='Density')
    ax.legend(fontsize=8)
fig.suptitle('Random LMDB records vs independent unique molecular identities (raw-SMILES)')
fig.tight_layout()
fig.savefig(RESULTS/'03_true_raw_vs_dedup_descriptor_distribution.png',dpi=200)
fig.savefig(RESULTS/'03_true_raw_vs_dedup_descriptor_distribution.pdf')
plt.show()

fig,ax=plt.subplots(figsize=(9,5))
focus=chemistry_direct.copy()
y=np.arange(len(focus))
err=np.vstack([focus['difference_unique_minus_raw']-focus['difference_ci_low'],
               focus['difference_ci_high']-focus['difference_unique_minus_raw']])
# Normalize each feature difference to the raw group SD for comparable display units.
delta=focus['standardised_difference'].to_numpy(float)
e=np.vstack([err[0]/focus['raw_sd'].to_numpy(),
             err[1]/focus['raw_sd'].to_numpy()])
ax.errorbar(delta,y,xerr=e,fmt='o',capsize=3)
ax.axvline(0,ls='--',color='gray',lw=1)
ax.axvspan(-.1,.1,color='gray',alpha=.10,label='Exploratory |SMD| < 0.1 region')
ax.set_yticks(y,focus['descriptor']);ax.invert_yaxis()
ax.set_xlabel('Difference (unique − raw), standardized by raw SD; bootstrap 95% CI')
ax.set_title('Descriptor differences and uncertainty, not a formal equivalence test')
ax.legend(loc='lower right',fontsize=9)
fig.tight_layout()
fig.savefig(RESULTS/'04_descriptor_effect_sizes_bootstrap.png',dpi=200)
fig.savefig(RESULTS/'04_descriptor_effect_sizes_bootstrap.pdf')
plt.show()

## 7 — Murcko scaffold diversity: did duplicate records distort **topology**, even if descriptors look similar?

Both datasets are randomly sampled independently. To compare richness fairly, **equal-size subsamples** of the two groups are drawn repeatedly. These values describe the raw-SMILES deduplication effect on *sampled scaffold exposure*, not on downstream task accuracy. Murcko scaffolds are crude for acyclic molecules and should not be treated as exhaustive structural diversity metrics.

In [ ]:
from collections import Counter
N_SC_REPEATS=200
N_SC=min(3000,len(raw),len(unique_real))
rng_sc=np.random.default_rng(SEED+881)
def scaffold_stats(vals):
    vc=pd.Series(vals).value_counts(normalize=True)
    entropy=-float((vc*np.log(vc)).sum())
    return {'distinct_scaffolds':len(vc),
            'effective_scaffolds_exp_entropy':np.exp(entropy),
            'top10_scaffold_share':float(vc.iloc[:10].sum()),
            'acyclic_share':float(vc.get('[acyclic]',0.0))}
scrows=[]
for group,frame in [('Raw',raw),('Dedup',unique_real)]:
    vals=frame['murcko_scaffold'].dropna().to_numpy(object)
    for rep in range(N_SC_REPEATS):
        x=rng_sc.choice(vals,size=N_SC,replace=False)
        scrows.append({'dataset':group,'repeat':rep,'sample_size':N_SC,**scaffold_stats(x)})
scaffold_df=pd.DataFrame(scrows)
sc_summary=scaffold_df.groupby('dataset').agg({
    col:['mean',lambda v:v.quantile(.025),lambda v:v.quantile(.975)]
    for col in ['distinct_scaffolds','effective_scaffolds_exp_entropy','top10_scaffold_share','acyclic_share']})
sc_summary.columns=[f'{a}_{b if b!="<lambda_0>" and b!="<lambda_1>" else ("q025" if b=="<lambda_0>" else "q975")}' for a,b in sc_summary.columns]
display(sc_summary.round(3))
scaffold_df.to_csv(RESULTS/'scaffold_equal_size_resample.csv',index=False)
sc_summary.to_csv(RESULTS/'scaffold_equal_size_summary.csv')
fig,axes=plt.subplots(1,2,figsize=(11,4))
for ax,feat,ttl in [(axes[0],'distinct_scaffolds','Distinct Murcko scaffolds'),
                     (axes[1],'effective_scaffolds_exp_entropy','Effective scaffold diversity')]:
    groupdata=[scaffold_df.loc[scaffold_df.dataset==lab,feat] for lab in ['Raw','Dedup']]
    ax.boxplot(groupdata,labels=['Raw','Actual dedup'],showfliers=False)
    ax.set(title=ttl,ylabel=f'Equal-size sample of {N_SC:,} molecules')
fig.suptitle('Scaffold exposure after removing repeated raw molecular identities')
fig.tight_layout()
fig.savefig(RESULTS/'05_scaffold_diversity.png',dpi=200)
fig.savefig(RESULTS/'05_scaffold_diversity.pdf')
plt.show()

## 8 — Direct 3D conformer evidence for *the same* SMILES in multiple LMDB records

This takes a **small random subset of duplicated identities** from the original raw sample, finds their two LMDB records using the *existing* raw-hash text export, and compares actual stored 3D geometries. It does **not** regenerate conformers, scan the LMDB, or replace any data.

Uses an atom-index-preserving **interatomic-distance RMSE** (Å), which is invariant to translation/rotation. Different conformers can have different values; a very small minimum indicates at least one matching geometry. This is exploratory evidence from sample pairs only; global 3D coverage is not inferred from 20 examples.

In [ ]:
from scipy.spatial.distance import pdist
N_PAIR_MAX=24
DUPLICATE_PAIRS_ENABLED=True
pair_rows=[]
if DUPLICATE_PAIRS_ENABLED:
    paired_digests=[h for h in PAIR_CANDIDATES if freq[h]>=2 and len(pair_export_ranks.get(h,[]))>=2]
    paired_digests=sorted(paired_digests)
    pair_rng=np.random.default_rng(SEED+444)
    pair_rng.shuffle(paired_digests)
    paired_digests=paired_digests[:N_PAIR_MAX]
    env=lmdb.open(str(TRAIN_LMDB),subdir=False,readonly=True,lock=False,readahead=False,meminit=False,max_readers=128)
    try:
        with env.begin(write=False) as txn:
            n=txn.stat()['entries']
            for h in paired_digests:
                keylist=[lex_rank_to_integer(rank,n) for rank in pair_export_ranks[h][:2]]
                records=[]
                for key in keylist:
                    v=txn.get(str(key).encode('ascii'))
                    assert v is not None, key
                    d=pickle.loads(v)
                    smi=d.get('smi',d.get(b'smi'))
                    if isinstance(smi,bytes):smi=smi.decode('utf8')
                    assert hash_smi(smi)==h,('hash mismatch',key)
                    records.append(d)
                atomseq=[tuple(str(a) for a in rec.get('atoms',rec.get(b'atoms',[]))) for rec in records]
                if atomseq[0] != atomseq[1] or len(atomseq[0])<2:
                    pair_rows.append(dict(digest=h,key_a=keylist[0],key_b=keylist[1],
                                          status='different atom order or too short',copies=freq[h]))
                    continue
                vectors=[]
                for d in records:
                    arr=[]
                    for coord in d.get('coordinates',d.get(b'coordinates',[])):
                        xyz=np.asarray(coord,dtype=float)
                        if xyz.shape != (len(atomseq[0]),3) or not np.isfinite(xyz).all(): continue
                        arr.append(pdist(xyz))
                    vectors.append(arr)
                if not vectors[0] or not vectors[1]:
                    pair_rows.append(dict(digest=h,key_a=keylist[0],key_b=keylist[1],
                                          status='no comparable coordinate slots',copies=freq[h]))
                    continue
                mat=np.array([[np.sqrt(np.mean((a-b)**2)) for b in vectors[1]] for a in vectors[0]])
                pair_rows.append(dict(digest=h,key_a=keylist[0],key_b=keylist[1],
                                      status='OK',copies=freq[h],
                                      atoms=len(atomseq[0]),nslots_a=len(vectors[0]),nslots_b=len(vectors[1]),
                                      min_cross_record_distance_rmse_A=float(mat.min()),
                                      median_cross_record_distance_rmse_A=float(np.median(mat)),
                                      fraction_cross_slot_pairs_below_0p02A=float(np.mean(mat<0.02))))
    finally: env.close()
    print('Actual duplicate molecule record pairs compared:',len(pair_rows))
paired=pd.DataFrame(pair_rows)
if not paired.empty:
    display(paired.drop(columns=['digest'],errors='ignore').head(12).round(4))
    paired.to_csv(RESULTS/'duplicate_record_3d_pair_check.csv',index=False)
    good=paired.loc[paired.status=='OK'].copy()
    if not good.empty:
        fig,ax=plt.subplots(figsize=(9,4.5))
        vals=good['min_cross_record_distance_rmse_A'].to_numpy(float)
        ax.hist(vals,bins=min(16,max(4,len(vals)//2)))
        ax.axvline(.02,color='gray',ls='--',label='Exploratory near-identical threshold (0.02 Å)')
        ax.set(xlabel='Minimum interatomic-distance RMSE between two records (Å)',
               ylabel='Number of duplicate identity pairs',
               title='Do same-SMILES LMDB records store different 3D geometry?')
        ax.legend(fontsize=9)
        fig.tight_layout()
        fig.savefig(RESULTS/'06_3d_geometry_between_duplicate_records.png',dpi=200)
        fig.savefig(RESULTS/'06_3d_geometry_between_duplicate_records.pdf')
        plt.show()
        print('Pairs with at least one nearly identical coordinate slot:',int((vals<.02).sum()),'/',len(vals))
        print('Note: each record may contain other conformers; minimum is NOT total conformer diversity.')

## 9 — One-page progress slide + results tables

This slide is generated from **measured outputs** and deliberately distinguishes three conclusions:
1. Raw molecules repeat (a fact).
2. Descriptors and scaffold distributions may change little (observations plus uncertainty).
3. *Retraining accuracy has not been tested.* Therefore these data **do not prove** deduplication has no impact on HEART quality.

The exported figure is a self-contained **PNG + PDF** for a progress presentation; use the output CSVs for details. Claims about conformer redundancy must come from the measured small paired check, not assumptions.

In [ ]:
from matplotlib.ticker import MaxNLocator
from matplotlib.gridspec import GridSpec
n_total=int(full['train']['total_records_scanned'])
n_unique=int(full['train']['unique_identifiers'])
n_excess=int(full['train']['duplicate_record_excess'])
fig=plt.figure(figsize=(13,7.3))
gs=GridSpec(2,3,figure=fig,height_ratios=[.64,1.0],hspace=.37,wspace=.38)
ax0=fig.add_subplot(gs[0,0]); ax1=fig.add_subplot(gs[0,1]); ax2=fig.add_subplot(gs[0,2]);
ax3=fig.add_subplot(gs[1,:2]); ax4=fig.add_subplot(gs[1,2])
ax0.bar(['Raw records','Raw-unique'],[n_total/1e6,n_unique/1e6])
ax0.set(ylabel='Millions',title='Full 18.84M audit')
ax0.text(.5,.9,f'{n_excess/n_total:.1%} duplicate excess',transform=ax0.transAxes,ha='center',fontsize=10)
ax1.bar(['Raw','Unique'],[100*(raw.copies_in_full_train>1).mean(),100*(uniq.copies_in_full_train>1).mean()])
ax1.set(title='Identities with >1 record',ylabel='% of sampled observations')
ax2.bar(['Raw','Unique'],[raw.copies_in_full_train.median(),uniq.copies_in_full_train.median()])
ax2.set(title='Median copies in original',ylabel='Copies');ax2.set_ylim(0,max(3,raw.copies_in_full_train.median()+1))
row=chemistry_direct.set_index('descriptor')
show=['heavy_atoms','molecular_weight','logP','TPSA','HBD','HBA','rotatable_bonds','ring_count']
y=np.arange(len(show)); dd=row.loc[show]
ax3.errorbar(dd.standardised_difference,y,
    xerr=np.vstack([dd.standardised_difference-(dd.difference_ci_low/dd.raw_sd),
                    dd.difference_ci_high/dd.raw_sd-dd.standardised_difference]),fmt='o',capsize=3)
ax3.axvline(0,color='gray',linestyle='--');ax3.axvspan(-.1,.1,alpha=.12,color='gray')
ax3.set_yticks(y,show);ax3.invert_yaxis()
ax3.set(xlabel='Unique − raw / raw SD (bootstrap 95% CI)',title='Descriptor shifts after deduplication')
ax4.boxplot([scaffold_df.loc[scaffold_df.dataset==lab,'effective_scaffolds_exp_entropy'] for lab in ['Raw','Dedup']],
            labels=['Raw','Unique'],showfliers=False)
ax4.set(title=f'Effective Murcko diversity\n(equal n={N_SC:,})',ylabel='Effective scaffold count')
fig.suptitle('HEART general ligand source: duplicate identity vs chemical-space exposure',fontsize=15,y=.98)
fig.text(.03,.022,'Data: exact full raw-SMILES audit + random samples; not an equivalence test. No retraining/performance ablation performed.',fontsize=9)
fig.subplots_adjust(top=.91,bottom=.10,left=.08,right=.98)
fig.savefig(RESULTS/'00_PROGRESS_SLIDE_duplicate_impact.png',dpi=220,bbox_inches='tight')
fig.savefig(RESULTS/'00_PROGRESS_SLIDE_duplicate_impact.pdf',bbox_inches='tight')
plt.show()

# Generate computed slide notes and a machine-readable report; no invented 'effect negligible' conclusion.
focus=['heavy_atoms','molecular_weight','logP','TPSA']
notes=[
    '# HEART duplicate impact — progress slide notes (measured)',
    f'- Full source: {n_total:,} raw records; {n_unique:,} raw-SMILES identities; {n_excess:,} excess copies ({n_excess/n_total:.2%}).',
    f'- Samples: {len(raw):,} randomly drawn raw records; {len(unique_real):,} independent actual unique identities.',
    f'- Frequency effect: {100*(raw.copies_in_full_train>1).mean():.1f}% of raw samples vs {100*(uniq.copies_in_full_train>1).mean():.1f}% of unique-identity samples originate from repeated molecules.',
    '- Independent chemistry mean shifts (unique minus raw; 95% bootstrap intervals):',
]
for feat in focus:
    r=row.loc[feat]
    notes.append(f"  - {feat}: {r.difference_unique_minus_raw:+.4f} [{r.difference_ci_low:+.4f}, {r.difference_ci_high:+.4f}]; SMD={r.standardised_difference:+.3f}; KS={r.KS_statistic:.3f}")
notes.extend([
    f'- Equal-n effective scaffold count (resample mean): Raw {scaffold_df.loc[scaffold_df.dataset=="Raw","effective_scaffolds_exp_entropy"].mean():.1f}; Dedup {scaffold_df.loc[scaffold_df.dataset=="Dedup","effective_scaffolds_exp_entropy"].mean():.1f}.',
    f'- Direct same-SMILES 3D record-pair audit: {len(paired)} sampled pairs (see CSV for atom-order and min-distance outcomes).',
    '- Scientific interpretation: identity duplication changes exposure frequency; whether it changes chemical breadth or geometry is evaluated descriptively by these random samples.',
    '- LIMITATION: this is **not** an experiment comparing checkpoints pretrained on raw vs deduplicated data, so it cannot prove deduplication would not improve HEART or lipid fine-tuning.',
    '- Decision: retain the 1M checkpoint for now; consider matched-compute ablation only if new performance/scaffold evidence warrants.',
])
(RESULTS/'progress_slide_notes.md').write_text('\n'.join(notes)+'\n',encoding='utf8')
report2={'n_full_raw':n_total,'n_full_unique_raw_smiles':n_unique,'n_full_excess_duplicate_records':n_excess,
    'sample_raw_records':len(raw),'independent_sample_actual_unique':len(unique_real),
    'descriptor_direct_comparison':chemistry_direct.replace({np.nan:None}).to_dict(orient='records'),
    'scaffold_equal_size':N_SC,'duplicate_pairs_inspected':len(paired),
    'no_pretraining_comparison_performed':True,
    'qualification':'These analyses cannot establish equivalent pretrained performance.'}
(RESULTS/'expanded_report.json').write_text(json.dumps(report2,indent=2,default=str)+'\n')
print('Output directory:',RESULTS)
print('Slide:',RESULTS/'00_PROGRESS_SLIDE_duplicate_impact.png')
print('Notes:',RESULTS/'progress_slide_notes.md')

## How to interpret this for General → Ugi-4CR → LNPDB training

- **If descriptor and scaffold shifts are small**, repeated raw-SMILES records are not obviously changing these *measured source distributions*. That reduces the urgency of discarding an already validated 1M checkpoint, **not** proof that the two training strategies perform identically.
- The planned Ugi-4CR continual stage and subsequent in-house lipid fine-tuning may attenuate or amplify effects of the original sampling distribution; neither is guaranteed. Retain the original checkpoint and a fixed downstream evaluation panel to detect a real difference.
- The most direct unresolved question is a same-compute **model ablation**; avoid launching a second 1M pretrain on histogram evidence alone.
- Original LMDB, hashes, trained checkpoints, previous notebooks and earlier result directories remain untouched.